# 05 — Structured Output
### LangChain Foundations

`04_Chains/` used a minimal `PydanticOutputParser` just to make one branch condition reliable. This
notebook is the full treatment: every way LangChain gets structured, program-usable data out of a
model instead of a paragraph you'd have to parse yourself.

## Why this topic exists at all

Everything up to this point has produced **text**. Text is great for a human to read, and useless
for a program to act on. "The sentiment is mostly positive, though there were complaints about
battery life" is a fine sentence — and completely unusable if you need `if sentiment == "negative":
send_to_support_queue()`. Structured output is the bridge between "the model said something
sensible" and "my code can act on exactly what it said."

## Setup

In [7]:
# %pip install -q langchain langchain-core langchain-groq python-dotenv pydantic
!pip install pydantic
!pip install pydantic[email]


   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ------------------------------------

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field, EmailStr
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("Environment ready.")


Environment ready.


---
## 1. Pydantic in 90 seconds (if you haven't used it before)

`with_structured_output()` leans on Pydantic heavily, so a 60-second primer pays for itself. A
`BaseModel` is a class that **validates its own data** the moment you create it — wrong type, missing
required field, value outside a constraint, and it raises immediately instead of failing silently
three steps later in your program.

In [13]:
class Student(BaseModel):
    name: str
    age: int
    city: Optional[str] = "Unknown"                                   # optional, with a default
    gender: Literal["f", "m", "other"]                                  # must be exactly one of these
    cgpa: float = Field(gt=0, lt=10, description="CGPA out of 10")      # validated numeric range
    email: EmailStr                                                     # validated email format

good_student = Student(name="Deepak", age=22, gender="m", cgpa=8.4, email="deepak@example.com")
print(good_student)
print(good_student.model_dump())   # -> a plain dict, once you need one


name='Deepak' age=22 city='Unknown' gender='m' cgpa=8.4 email='deepak@example.com'
{'name': 'Deepak', 'age': 22, 'city': 'Unknown', 'gender': 'm', 'cgpa': 8.4, 'email': 'deepak@example.com'}


In [14]:
# Validation actually runs — this is the entire point of using Pydantic over a plain dict.
try:
    Student(name="Test", age=22, gender="x", cgpa=15, email="not-an-email")
except Exception as e:
    print("Validation caught it:")
    print(e)


Validation caught it:
3 validation errors for Student
gender
  Input should be 'f', 'm' or 'other' [type=literal_error, input_value='x', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
cgpa
  Input should be less than 10 [type=less_than, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]


---
## 2. `with_structured_output()` — the modern, provider-native approach

This is what you'll reach for by default. Give it a schema (a `TypedDict` or a Pydantic `BaseModel`),
and the model returns data matching that exact shape — under the hood, most providers implement this
using their native function/tool-calling capability, which is far more reliable than hoping the model
formats JSON correctly on its own.

### Option A: `TypedDict` — fast, lightweight, no runtime validation

Good for quick extraction tasks where you trust the model's output and don't need Pydantic's
validation guarantees. Notice: you get a plain `dict` back, accessed with `result['key']`.

In [15]:
class ReviewTD(TypedDict):
    key_themes: Annotated[list[str], "Key themes discussed in the review"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["positive", "negative", "neutral"], "Overall sentiment"]
    pros: Annotated[Optional[list[str]], "Pros mentioned in the review"]
    cons: Annotated[Optional[list[str]], "Cons mentioned in the review"]

structured_model_td = model.with_structured_output(ReviewTD)

sample_review = """
I ordered this gaming laptop after reading dozens of positive reviews. It arrived on time and the
build quality is fantastic. Games run smoothly even on ultra settings, and the keyboard feels premium.
Unfortunately, after only two weeks the cooling fans started making loud noises. Customer support
responded quickly and arranged a replacement, but the replacement unit had the same issue. Battery
life is also disappointing, lasting barely three hours under normal use.
"""

result = structured_model_td.invoke(sample_review)
print(type(result))          # dict
print(result["sentiment"])   # dict-style access
print(result["pros"])


<class 'dict'>
negative
['Fantastic build quality', 'Games run smoothly even on ultra settings', 'Premium-feeling keyboard', 'Quick and helpful customer support']


### Option B: Pydantic `BaseModel` — validated, real objects

Same idea, but you get an actual `Review` instance back — attribute access (`result.sentiment`, not
`result['sentiment']`), and every field is validated against the type you declared. Use this whenever
the structured data is going anywhere important (a database, a business decision, another API call).

In [16]:
class Review(BaseModel):
    key_themes: list[str] = Field(description="Key themes discussed in the review")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="Overall sentiment")
    pros: Optional[list[str]] = Field(default=None, description="Pros mentioned in the review")
    cons: Optional[list[str]] = Field(default=None, description="Cons mentioned in the review")
    reviewer_name: Optional[str] = Field(default=None, description="The reviewer's name, if mentioned")

structured_model = model.with_structured_output(Review)

result = structured_model.invoke(sample_review)
print(type(result))          # Review — a real, validated object
print(result.sentiment)      # attribute access
print(result.pros)


<class '__main__.Review'>
negative
['Fantastic build quality', 'Games run smoothly on ultra settings', 'Premium-feeling keyboard', 'On‑time delivery', 'Responsive customer support']


---
## 3. `TypedDict` vs. Pydantic vs. `PydanticOutputParser` — when to use which

| Approach | Validation? | Output type | Needs provider function-calling support? | Use when |
|---|---|---|---|---|
| `with_structured_output(TypedDict)` | ❌ No | `dict` | ✅ Yes | Quick extraction, you trust the model, speed matters more than guarantees |
| `with_structured_output(BaseModel)` | ✅ Yes | Pydantic object | ✅ Yes | Anything feeding a real system — the default choice for production |
| `PydanticOutputParser` (from `04_Chains`) | ✅ Yes | Pydantic object | ❌ No — works via prompt instructions alone | A provider/model that doesn't support structured output natively, or you need it to work through a plain completion-style model |

**In practice:** default to `with_structured_output(BaseModel)`. Only drop to `PydanticOutputParser`
when you've confirmed a provider genuinely doesn't support the native method — it's more fragile
(the model has to *follow instructions* to format correctly, rather than the provider *enforcing*
the shape), so it's a fallback, not a first choice.

---
## 4. Nested models — extracting structured lists of structured things

Real extraction tasks often aren't flat. A review might mention several distinct products or
features, each with its own sentiment. Pydantic models can nest inside each other — the schema you
declare can be as detailed as the data actually is.

In [17]:
class ProductMention(BaseModel):
    feature: str = Field(description="The specific product feature or aspect mentioned")
    sentiment: Literal["positive", "negative", "neutral"]

class DetailedReview(BaseModel):
    overall_sentiment: Literal["positive", "negative", "neutral"]
    summary: str
    mentions: list[ProductMention] = Field(description="Each distinct feature/aspect mentioned, with its own sentiment")

detailed_model = model.with_structured_output(DetailedReview)

result = detailed_model.invoke(sample_review)
print("Overall:", result.overall_sentiment)
for m in result.mentions:
    print(f"  - {m.feature}: {m.sentiment}")


Overall: negative
  - build quality: positive
  - gaming performance: positive
  - keyboard: positive
  - cooling fans: negative
  - customer support: positive
  - replacement unit issue: negative
  - battery life: negative


---
## Real-world capstone: batch review analysis + aggregate report

This is where structured output actually pays off — not printing one pretty result, but running it
across many inputs with `.batch()` and then doing **real Python logic** on the results: something
you could never safely do on raw paragraphs of text.

In [18]:
reviews = [
    "Amazing battery life and the camera is stunning in low light. Software feels a bit laggy though.",
    "Terrible experience. It overheated within a day and support never responded to my emails.",
    "Solid mid-range phone. Nothing exceptional, nothing bad. Does what I need it to do.",
    "The screen is gorgeous and it's fast, but the price is honestly too high for what you get.",
    "Worst purchase this year. Battery died within 6 months and replacement was refused.",
]

results = structured_model.batch(reviews)   # one call, all five reviews processed

sentiment_counts = {"positive": 0, "negative": 0, "neutral": 0}
all_cons = []

for r in results:
    sentiment_counts[r.sentiment] += 1
    if r.cons:
        all_cons.extend(r.cons)

print("Sentiment breakdown:", sentiment_counts)
print(f"\n{sentiment_counts['negative']}/{len(reviews)} reviews were negative "
      f"({sentiment_counts['negative'] / len(reviews):.0%})")
print("\nAll complaints mentioned across every review:")
for c in all_cons:
    print(" -", c)


Sentiment breakdown: {'positive': 0, 'negative': 3, 'neutral': 2}

3/5 reviews were negative (60%)

All complaints mentioned across every review:
 - Software feels laggy
 - overheating
 - lack of support response
 - price is too high for what you get


Notice what just happened: five independent paragraphs of free-form text turned into one
aggregate statistic (`"40% of reviews were negative"`) and a clean complaint list — computed with
plain Python `dict` counting, not another model call. That's the entire value of structured output:
it's the point where an LLM's output stops being "text to read" and starts being "data to compute on."

### Try it yourself
1. Add `reviewer_name` extraction back in and print a table of `name -> sentiment` for reviews that
   mention one.
2. Change `sentiment` from a 3-way `Literal` to a 1-5 star rating (`Literal[1,2,3,4,5]`) and see how
   the model adapts.
3. Deliberately feed `structured_model` a review in a language the model handles less well, and check
   whether the Pydantic validation ever fails — this is exactly the kind of edge case
   `16_Error_Handling/` (later in this repo) teaches you to guard against with `.with_retry()`.
4. Try `method="json_mode"` as an argument to `with_structured_output()` and compare — some
   providers support both function-calling-based and native JSON-mode-based structured output, and
   they don't always produce identical results.

---
**Next:** `06_Document_Loaders/` — a shift in direction: instead of shaping a model's *output*,
you start feeding it real external content (files, web pages, PDFs) as *input*.
